In [ ]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning) # silences future development warninigs

import earthaccess
import folium
import geopandas as gpd
import numpy as np
np.seterr(divide='ignore', invalid='ignore') # silence maths warnings (0/0 or X/0) globally
import pandas as pd
from pathlib import Path
import rasterio as rio
import rasterio.merge
from rasterio import features
from rasterio.merge import merge
from rasterio.windows import from_bounds
from rasterstats import zonal_stats
import rioxarray
import shapely
from shapely.geometry import shape
import xarray as xr
from xrspatial import hillshade
import matplotlib.pyplot as plt


def get_bands_by_satellite():
    records = []

    # Map colors to band numbers based on the Satellite ID prefix
    # LT04/05 = Landsat 4/5 TM; LE07 = Landsat 7 ETM+; LC08/09 = Landsat 8/9 OLI
    band_config = {
    "LT05":{"B1":"BLUE", "B2":"GREEN", "B3":"RED", "B4":"NIR", "B5":"SWIR1", "B7":"SWIR2"},
    "LT07":{"B1":"BLUE", "B2":"GREEN", "B3":"RED", "B4":"NIR", "B5":"SWIR1", "B6":"TIR", "B7":"SWIR2" },
    "LC08":{"B1":"COAST/AERO", "B2":"BLUE", "B3":"GREEN", "B4":"RED", "B5":"NIR", "B6":"SWIR1", "B7":"SWIR2"},
    "LC09":{"B1":"COAST/AERO", "B2":"BLUE", "B3":"GREEN", "B4":"RED", "B5":"NIR", "B6":"SWIR1", "B7":"SWIR2"}
}
    for folder in PATHS["landsat_images"].iterdir():
        if folder.is_dir():
            for file in folder.glob("*_SR_B*.TIF"):
                parts = file.name.split("_")
                sat_id = parts[0][:4]
                band_id = parts[-1].replace(".TIF", "")
                if band_config[sat_id][band_id] in ALL_COLOURS:
                    records.append({
                        "satellite": sat_id,
                        "path_row": parts[2],
                        "year": parts[3][:4],
                        "band": band_id,
                        "colour": band_config[sat_id][band_id],
                        "filename": file.name,
                        "path": str(file)
                    })
    return pd.DataFrame(records)



def create_mosaic(file_list, out_path, dtype=None): 
    # 1. Get CRS from the first file for comparison with other files
    with rio.open(file_list[0]) as src:
        target_crs = src.crs
        out_meta = src.profile.copy()

    # 2. Compare othes files against ref CRS and reproject if they don't match
    processed_list = []
    for f in file_list:
        with rio.open(f) as src:
            if src.crs != target_crs:
                print(f"Fixing CRS for: {f.name}")
                processed_list.append(fix_projection(f, target_crs))
            else:
                processed_list.append(f)
                
    # 3. Merge images, return the new array and new spatial transform
    #print(f"    Mosaicking stared: {out_path.name}")
    mosaic, out_trans = merge(processed_list, nodata=0)

    # 4. Update the metadata with mosiac dimensions and transform
    bands, height, width = mosaic.shape
    out_meta.update({
        "height": height, 
        "width": width, 
        "transform": out_trans, 
        "nodata": 0,
        "dtype": dtype or out_meta['dtype'] # And here
    })

    # 5. Save mosaic and full metadata to file
    out_path.parent.mkdir(parents=True, exist_ok=True)
    with rio.open(out_path, "w", **out_meta) as dest:
        dest.write(mosaic)

    print(f"     Mosaic saved: {out_path.name}")



def ndi_and_clipping(yr, border_gdf):
    """
    1. Loads windowed mosaic bands for the year.
    2. Stacks them into an Xarray Dataset.
    3. Clips the stack to the jagged park boundary.
    4. Calculates and saves each NDI index.
    """
    raster_data = {}
    win_meta = None
    
    for band in ANALYSIS_COLOURS:
        file = PATHS["mosaics"] / f"{yr}_{band}_mosaic.tif"
        with rio.open(file) as src:
            if border_gdf.crs != src.crs:
                border_gdf = border_gdf.to_crs(src.crs)
                
            window = from_bounds(*border_gdf.total_bounds, transform = src.transform)
            raster_data[band] = xr.DataArray(src.read(1,window=window).astype("float32"), dims=("y","x"))#flaot32 added here to support the  ndvi calulations later on
            if win_meta is None:
                win_meta = {'crs':src.crs, 'transform':src.window_transform(window)}

    ds = xr.Dataset(raster_data)
    ds = ds.rio.write_crs(win_meta['crs']).rio.write_transform(win_meta['transform'])

    ds_clipped = ds.rio.clip(border_gdf.geometry, drop =True)
            
    for NDI in ANALYSIS_TASKS:
        b1_name, b2_name = INDEX_TO_BANDS[NDI]
        b1, b2 = ds_clipped[b1_name], ds_clipped[b2_name]   

        result = (b1 - b2) / (b1 + b2)
        result = result.where(~np.isinf(result)) # Replaces Inf with NaN
        
        ndi_out = PATHS["ndi"] / f"{yr}_{NDI}.tif"
        result.rio.to_raster(ndi_out, nodata=np.nan)
        print(f"     Processed and saved: {yr}_{NDI}.tif")
    del result, ds_clipped, raster_data


# creating composite images
def normalize(band):
    #data_only = band[band>0]
    #vmin, vmax = np.nanpercentile(band, [1, 99])
    vmax, vmin = band.max(), band.min()
    norm = np.clip(band, vmin, vmax)
    norm = ((norm - vmin)/(vmax - vmin))
   #norm = np.power(norm, 3) 
#    norm[band == 0] = 0
    return norm
   
def composite_images(yr, disp_type, input_path, dest_path):
        path1 = input_path / f"{yr}_{INDEX_TO_BANDS[disp_type][0]}_mosaic.tif"
        path2 = input_path / f"{yr}_{INDEX_TO_BANDS[disp_type][1]}_mosaic.tif"
        path3 = input_path / f"{yr}_{INDEX_TO_BANDS[disp_type][2]}_mosaic.tif"
    
        with rio.open(path1) as src1, rio.open(path2) as src2, rio.open(path3) as src3:
            colr1 = src1.read(1)
            colr2 = src2.read(1)
            colr3 = src3.read(1)
            out_meta = src1.meta.copy()
            ncolr1 = (normalize(colr1) * 255).astype(np.uint8)
            ncolr2 = (normalize(colr2) * 255).astype(np.uint8)
            ncolr3 = (normalize(colr3) * 255).astype(np.uint8)
            
            out_meta.update({
                "driver": "GTiff",
                "count": 3,
                "dtype": "uint8",
                "compress": "lzw"
            })
            out_filename = dest_path / f"{yr}_{disp_type}_composite.tif"

            with rio.open(out_filename, 'w', **out_meta) as dst:
                dst.write(ncolr3, 1) # Red channel (or first selected band)
                dst.write(ncolr2, 2) # Green channel
                dst.write(ncolr1, 3) # Blue channel
                
            print(f"     Saved: {out_filename.name}")

# --- 1. User Inputs ---
base = Path("C:/RS_GIS/EGM722/Assignment/Grand_Canyon/USGS_data/Unzipped") # insert path of base directory into brackets, eg. Path(<base_folder_location>)
boundary_file = Path("C:/RS_GIS/EGM722/Assignment/Grand_Canyon/USGS_data/Unzipped/Boundary Files/national_park_boundary.shp")

#select desired display and analysis: 0 = not wanted, 1 = wanted.
USER_OPTIONS = {
    "TRUE_COLOUR": 1,
    "FALSE_COLOUR": 0,
    "NDVI": 1,
    "NDWI": 1,
    "NDSI": 0,
}
print("1. User Inputs and selections accepted")

# --- 2. Creating folder structure ---
PATHS = {
    "landsat_images" : base / "Landsat_Images",
    "ndi": base / "NDI Images",
    "mosaics": base / "Mosaics",
    "boundaries": base / "Boundary Files",
    "earthaccess": base / "EarthAccess",
    "results": base / "Results"
}
for p in PATHS.values(): p.mkdir(parents=True, exist_ok=True)
print("2. Directories checked/created")

# --- 3. Determine colour bands requried for selected disaply / analysis 
INDEX_TO_BANDS = {
    "TRUE_COLOUR" : ["BLUE", "GREEN", "RED"],
    "FALSE_COLOUR" : ["GREEN","RED","NIR"],
    "NDVI": ["RED", "NIR"],
    "NDWI": ["GREEN", "NIR"],
    "NDSI": ["GREEN", "SWIR1"]
}

DISPLAY_TASKS = [task for task in ["TRUE_COLOUR", "FALSE_COLOUR"] if USER_OPTIONS.get(task)]
display_colours_raw = list({band for key in DISPLAY_TASKS for band in INDEX_TO_BANDS[key]})
colour_order = ["COAST/AERO","BLUE", "GREEN", "RED", "NIR", "SWIR1", "SWIR2"]
DISPLAY_COLOURS = [b for b in colour_order if b in display_colours_raw]
ANALYSIS_TASKS = [task for task in ["NDVI", "NDWI", "NDSI"] if USER_OPTIONS.get(task)]
ANALYSIS_COLOURS = list({band for key in ANALYSIS_TASKS for band in INDEX_TO_BANDS[key]})
ALL_COLOURS = list(set(DISPLAY_COLOURS + ANALYSIS_COLOURS))
print("3. Band identification and mapping completed")

# --- 4. Build DataFrame of rasters needed to create mosaicks (later an xarray will be usilt to laocte the mosaics for analysing) ---
df = get_bands_by_satellite()
df.to_csv(PATHS["landsat_images"] / "bands_dataframe.csv", index = False)
print("4. Lansdat image dataset created")


# --- 5. Acquiring DEMs ---
border_poly_proj = border_gdf.to_crs(epsg = 4326).union_all() #projection EPSG 4326 needed for earthacess
search_area = border_poly_proj.minimum_rotated_rectangle
search_area = shapely.geometry.polygon.orient(search_area, sign=1)
#getting dem data from earthaccess
print("7. EarthAccess data acquistion started...")
earthaccess.login(strategy='netrc')
earthaccess_files = earthaccess.search_data(
    short_name = 'ASTGTM',
    polygon = search_area.exterior.coords
)
if len(earthaccess_files) == 0:
    print("  No Data Found in Earth Access")
    #code here to stop whole program?
    
earthaccess_dest = PATHS["earthaccess"]
downloaded_files = earthaccess.download(earthaccess_files, earthaccess_dest)


# --- 6. create hillshde
hillshade_orig = hillshade(dem_ds, azimuth=225, angle_altitude=45)
hillshade_26912 = hillshade_orig.rio.reproject("EPSG:26912")
hillshade_dest = PATHS["results"] / "hillshade.tif"
hillshade_26912.rio.to_raster(hillshade_dest, compress = "lzw")



# --- 7. Mosaicking rasters---
years = df["year"].unique()
print("5. Ratser mosaicking started...")
for yr in years:
    for colr in ALL_COLOURS:
        file_list = df[(df["year"] == yr) & (df["colour"] == colr)]["path"].tolist()
        dst_path = PATHS["mosaics"] / f"{yr}_{colr}_mosaic.tif"
        create_mosaic(file_list, dst_path)
print("     All rasters mosaicked")

# --- 8. Mosaicking DEM---
print("8. DEM mosaicking started...")
dem_files = [fn for fn in downloaded_files if 'dem.tif' in fn.name]
dem_mosaic_dest = PATHS["earthaccess"] / "DEM_mosaic.TIF"
dem_meta = create_mosaic(dem_files, dem_mosaic_dest, dtype='float32')


# --- Create xarray of Raster and mosaics DEMs ----


        #### need the code here ###

# --- 9. NDVI analysis and image clipping---
print("6. Normalised difference analysis and clipping started...")
border_gdf = gpd.read_file(boundary_file)
for yr in years:
    ndi_and_clipping(yr, border_gdf)
print("     Calculations and clipping completed")

# --- ?. Can't remeber what this does - Google AI, pelase enlightne me! ---
with rio.open(dem_mosaic_dest) as src:
    border_gdf_proj = border_gdf.to_crs(src.crs)
    
    window = from_bounds(*border_gdf_proj.total_bounds, transform = src.transform)
    dem_windowed = xr.DataArray(src.read(1,window=window).astype("float32"), dims=("y","x"))
    win_meta = {'crs':src.crs, 'transform':src.window_transform(window)}

dem_ds = xr.Dataset({"elevation":dem_windowed})
dem_ds = dem_ds.rio.write_crs(win_meta['crs']).rio.write_transform(win_meta['transform'])
dem_ds_clipped = dem_ds.rio.clip(border_gdf_proj.geometry, drop =True)
print("9. DEM clipped to park boundary")


# --- 10. Vectorising clipped DEM ---
print("10. DEM Vectorisation started...")
step = 250
dem_ds_classified = (dem_ds_clipped.elevation.values // step).astype("int32")
mask = ~np.isnan(dem_ds_clipped.elevation.values)
shapes = features.shapes(dem_ds_classified, mask=mask,transform=dem_ds_clipped.rio.transform())
polygons = []
for i, (geom, value) in enumerate(shapes):
    zone_label = int(value*step)
    polygons.append({
        'poly_id':i,
        'elevation_zone': zone_label,
        'geometry':shape(geom)
    })

segments_gdf = gpd.GeoDataFrame(polygons, crs=dem_ds_clipped.rio.crs)
print("      Vectorising completed")


# --- 11. Caclualting Zonal Statistics ---
print("11. Calculating Zonal Statistics...")
ndvi_path = PATHS['ndi']/"2020_NDVI.tif" # path for NDVI image to be cmopared against segmented DEM
stats = zonal_stats(segments_gdf,ndvi_path,stats='mean') # Runing zonal statistics on ndvi_path
segments_gdf['Mean NDVI'] = [s['mean'] for s in stats] # Attaching results to GeoDataFrame
segments_gdf = segments_gdf.dropna(subset = 'Mean NDVI') # cleaning up empty or NoData zones
segments_gdf.to_csv(PATHS['results']/"zonal_statistics.csv", index=False)
print("      Calculated")

# --- 12. Creating images for dispaly ---
print("12. Creating display images...")
for yr in years:
    for disp in DISPLAY_TASKS:
        composite_images(yr, disp, PATHS["mosaics"], PATHS["results"])


print("      ALL images created ")

print("--- All Tasks Completed ---")

